# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dilip-chendra/FlyRank-Week1/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## Method Choice and Why

For this assignment I selected a Random Forest Classifier.

Random Forest can learn relationships between multiple search performance signals without relying on fixed thresholds. It is more flexible than the Week 4 rule-based baseline while still allowing feature importance analysis.

The goal is to compare the machine learning model against the baseline using the same dataset and an honest train/test split.

In [4]:
import os
import sys
import subprocess
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )
    os.chdir(REPO_DIR)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## Split Design

An 80/20 train-test split is used.

The training set is used to fit the model and the test set is kept separate for evaluation. This provides an honest estimate of performance on unseen data.

The same dataset is used for both the baseline comparison and the machine learning model so that the comparison is fair.

In [3]:
from sklearn.model_selection import train_test_split

features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "search_volume",
    "word_count"
]

X = df[features].fillna(0)

target = (df["trend_direction"] == "down").astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    target,
    test_size=0.20,
    random_state=42
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 24000
Testing rows: 6000


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## Train and Compare with the Baseline

A Random Forest model is trained using search performance features.

The model is evaluated using Accuracy, Precision, Recall, and F1 Score. These metrics are compared with the Week 4 baseline to determine whether the machine learning model provides better predictive performance.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)
f1 = f1_score(y_test, pred)

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Random Forest": [
        round(accuracy, 3),
        round(precision, 3),
        round(recall, 3),
        round(f1, 3)
    ]
})

results

,Metric,Random Forest
0,Accuracy,0.601
1,Precision,0.624
2,Recall,0.674
3,F1 Score,0.648


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Errors and Interpretation

The Random Forest model combines multiple signals instead of relying on fixed thresholds.

Some errors are expected because search performance can change due to seasonality, recent updates, or factors that are not available in the dataset.

Feature importance helps explain which variables the model relied on most heavily.

In [6]:
importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
})

importance = importance.sort_values(
    "Importance",
    ascending=False
)

print("Feature Importance")
print(importance)

comparison = X_test.copy()
comparison["Actual"] = y_test.values
comparison["Predicted"] = pred

errors = comparison[
    comparison["Actual"] != comparison["Predicted"]
]

print("\nSample Errors")
errors.head(10)

Feature Importance
           Feature  Importance
0  impressions_90d    0.417403
4       word_count    0.297535
2              ctr    0.117512
3    search_volume    0.084838
1       clicks_90d    0.082711

Sample Errors


,impressions_90d,clicks_90d,ctr,search_volume,word_count,Actual,Predicted
2308,283,0,0.00,0.0,1397.0,0,1
22404,8878,8,0.09,0.0,3188.0,0,1
7790,8,2,25.00,0.0,845.0,1,0
10784,12291,34,0.28,70.0,0.0,0,1
14474,32029,157,0.49,30.0,3166.0,1,0
28694,1251,5,0.40,30.0,2532.0,1,0
14054,52,0,0.00,10.0,3518.0,0,1
9482,134567,86,0.06,0.0,3955.0,0,1
15107,3554,4,0.11,20.0,0.0,1,0
5937,286,0,0.00,0.0,0.0,0,1


## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card.